# 03 - Analyze Results

This notebook analyzes LLM performance on the Backtest Lie Detector benchmark.

## Contents
1. Load results and benchmark
2. Aggregate metrics and leaderboard
3. Module-level analysis
4. Violation-level analysis
5. Failure taxonomy
6. Confidence calibration
7. Example failures
8. Export figures and report

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from backtest_lie_detector.schemas import Module, Difficulty, ViolationType
from backtest_lie_detector.benchmark.build_cases import load_benchmark
from backtest_lie_detector.evals.run_eval import load_results
from backtest_lie_detector.evals.scoring import (
    aggregate_scores,
    aggregate_by_module,
    aggregate_by_violation_type,
    scores_to_dataframe,
    SEVERITY_WEIGHTS,
)
from backtest_lie_detector.analysis.plots import (
    plot_leaderboard,
    plot_module_breakdown,
    plot_violation_heatmap,
    plot_confidence_vs_accuracy,
    plot_difficulty_breakdown,
    generate_all_plots,
)
from backtest_lie_detector.analysis.failure_taxonomy import (
    categorize_all_failures,
    generate_failure_report,
    get_failure_statistics,
    sample_failures_by_category,
    FailureCategory,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
# Paths
BENCHMARK_PATH = "../data/benchmark/benchmark_v1.jsonl"
RESULTS_PATH = "../outputs/results/model_outputs.jsonl"
FIGURES_DIR = "../outputs/figures"

# Load benchmark
cases = load_benchmark(BENCHMARK_PATH)
cases_df = pd.DataFrame([{
    'id': c.id,
    'module': c.module.value,
    'difficulty': c.difficulty.value,
    'expected_validity': c.expected_validity.value,
    'expected_violations': ','.join(v.value for v in c.expected_violations),
    'n_violations': len(c.expected_violations),
} for c in cases])

print(f"Loaded {len(cases)} benchmark cases")
cases_df.head()

In [ ]:
# Load results
try:
    scores = load_results(RESULTS_PATH)
    scores_df = scores_to_dataframe(scores)
    print(f"Loaded {len(scores)} scored responses")
    print(f"Configurations: {scores_df['config_name'].unique().tolist()}")
except FileNotFoundError:
    print("No results found. Please run 02_run_evals.ipynb first.")
    print("\nCreating mock results for demonstration...")
    
    # Create mock results for demonstration
    mock_data = []
    for case in cases:
        for config in ['mock_finance_auditor', 'mock_minimal']:
            # Simulate some results
            correct = np.random.random() > 0.3  # 70% accuracy
            mock_data.append({
                'case_id': case.id,
                'config_name': config,
                'model_name': 'mock-model',
                'parse_success': True,
                'validity_correct': correct,
                'violation_precision': np.random.random() * 0.4 + 0.5,
                'violation_recall': np.random.random() * 0.4 + 0.4,
                'violation_f1': np.random.random() * 0.4 + 0.4,
                'severity_weighted_recall': np.random.random() * 0.4 + 0.4,
                'repair_score': np.random.random() * 0.5 + 0.3,
                'confidence': np.random.random() * 0.4 + 0.5,
                'latency_ms': np.random.random() * 500 + 500,
                'predicted_validity': 'valid' if correct == (case.expected_validity.value == 'valid') else 'invalid',
                'predicted_violations': ','.join(v.value for v in case.expected_violations[:1]) if not correct else '',
            })
    
    scores_df = pd.DataFrame(mock_data)
    print(f"Created {len(scores_df)} mock results")
    
    # Also create mock ScoredResponse objects
    scores = []

## 2. Leaderboard

In [ ]:
# Aggregate metrics by configuration
leaderboard = scores_df.groupby('config_name').agg({
    'parse_success': 'mean',
    'validity_correct': 'mean',
    'violation_precision': 'mean',
    'violation_recall': 'mean',
    'violation_f1': 'mean',
    'severity_weighted_recall': 'mean',
    'repair_score': 'mean',
    'latency_ms': 'mean',
}).round(3)

leaderboard.columns = [
    'Parse Rate', 'Validity Acc', 'Violation Prec', 'Violation Recall',
    'Violation F1', 'Severity-Weighted', 'Repair Score', 'Latency (ms)'
]

# Sort by validity accuracy
leaderboard = leaderboard.sort_values('Validity Acc', ascending=False)
leaderboard

In [ ]:
# Plot leaderboard
fig = plot_leaderboard(scores_df, title="Model Leaderboard: Validity Accuracy")
plt.show()

## 3. Module-Level Analysis

In [ ]:
# Merge scores with case info
merged = scores_df.merge(cases_df[['id', 'module', 'difficulty']], 
                          left_on='case_id', right_on='id', how='left')

# Accuracy by module
module_acc = merged.groupby(['config_name', 'module'])['validity_correct'].mean().unstack()
module_acc.round(3)

In [ ]:
# Plot module breakdown
fig = plot_module_breakdown(scores_df, cases_df, title="Accuracy by Benchmark Module")
plt.show()

In [ ]:
# Key findings by module
print("\n=== Key Findings by Module ===\n")

for module in Module:
    module_data = merged[merged['module'] == module.value]
    if len(module_data) == 0:
        continue
    
    mean_acc = module_data['validity_correct'].mean()
    best_config = module_data.groupby('config_name')['validity_correct'].mean().idxmax()
    best_acc = module_data.groupby('config_name')['validity_correct'].mean().max()
    
    print(f"{module.value.replace('_', ' ').title()}:")
    print(f"  Average accuracy: {mean_acc:.1%}")
    print(f"  Best config: {best_config} ({best_acc:.1%})")
    print()

## 4. Difficulty Analysis

In [ ]:
# Accuracy by difficulty
diff_acc = merged.groupby(['config_name', 'difficulty'])['validity_correct'].mean().unstack()
diff_acc = diff_acc[['easy', 'medium', 'hard']]  # Order columns
diff_acc.round(3)

In [ ]:
# Plot difficulty breakdown
fig = plot_difficulty_breakdown(scores_df, cases_df, title="Accuracy by Difficulty Level")
plt.show()

## 5. Violation-Level Analysis

In [ ]:
# Violation recall by type
print("\n=== Violation Detection by Type ===\n")
print(f"{'Violation Type':<35} {'Count':<8} {'Severity':<10}")
print("-" * 55)

for vtype in ViolationType:
    count = sum(1 for c in cases if vtype in c.expected_violations)
    weight = SEVERITY_WEIGHTS.get(vtype, 1)
    print(f"{vtype.value:<35} {count:<8} {weight:<10}")

In [ ]:
# Try to plot violation heatmap (requires predicted_violations column)
if 'predicted_violations' in scores_df.columns:
    try:
        fig = plot_violation_heatmap(scores_df, cases_df, title="Violation Detection Recall by Type")
        plt.show()
    except Exception as e:
        print(f"Could not generate heatmap: {e}")

## 6. Confidence Calibration

In [ ]:
# Confidence analysis
if 'confidence' in scores_df.columns:
    conf_analysis = scores_df.groupby('config_name').agg({
        'confidence': 'mean',
        'validity_correct': 'mean',
    }).round(3)
    
    # Add confidence when correct/wrong
    for config in scores_df['config_name'].unique():
        cfg_data = scores_df[scores_df['config_name'] == config]
        conf_analysis.loc[config, 'conf_when_correct'] = cfg_data[cfg_data['validity_correct']]['confidence'].mean()
        conf_analysis.loc[config, 'conf_when_wrong'] = cfg_data[~cfg_data['validity_correct']]['confidence'].mean()
    
    conf_analysis.columns = ['Mean Confidence', 'Accuracy', 'Conf When Correct', 'Conf When Wrong']
    conf_analysis.round(3)

In [ ]:
# Plot confidence vs accuracy
if 'confidence' in scores_df.columns:
    fig = plot_confidence_vs_accuracy(scores_df, title="Confidence Calibration")
    plt.show()

## 7. Failure Analysis

In [ ]:
# Get failure statistics
if scores:  # If we have actual ScoredResponse objects
    failure_stats = get_failure_statistics(cases, scores)
    
    print("\n=== Failure Statistics ===\n")
    print(f"Total responses: {failure_stats['total_responses']}")
    print(f"Parse success rate: {failure_stats['parse_success_rate']:.1%}")
    print(f"Failure rate: {failure_stats['failure_rate']:.1%}")
    
    print("\nFailures by category:")
    for cat, count in failure_stats['by_category'].items():
        if count > 0:
            print(f"  {cat}: {count}")
else:
    # Approximate from DataFrame
    print("\n=== Failure Analysis (from DataFrame) ===\n")
    
    total = len(scores_df)
    correct = scores_df['validity_correct'].sum()
    wrong = total - correct
    
    print(f"Total responses: {total}")
    print(f"Correct: {correct} ({correct/total:.1%})")
    print(f"Incorrect: {wrong} ({wrong/total:.1%})")
    
    # High confidence errors
    if 'confidence' in scores_df.columns:
        overconf = scores_df[(~scores_df['validity_correct']) & (scores_df['confidence'] > 0.7)]
        print(f"\nOverconfident errors (confidence > 70%): {len(overconf)}")

In [ ]:
# Show example failures
print("\n=== Example Failures ===\n")

failures = merged[~merged['validity_correct']].head(5)

for _, row in failures.iterrows():
    case = next((c for c in cases if c.id == row['case_id']), None)
    if case:
        print(f"Case: {row['case_id']}")
        print(f"  Module: {row['module']}")
        print(f"  Difficulty: {row['difficulty']}")
        print(f"  Config: {row['config_name']}")
        print(f"  Prompt: {case.prompt[:100]}...")
        print(f"  Expected: {case.expected_validity.value}")
        print(f"  Predicted: {row.get('predicted_validity', 'N/A')}")
        print(f"  Confidence: {row.get('confidence', 'N/A'):.0%}" if pd.notna(row.get('confidence')) else "  Confidence: N/A")
        print()

## 8. Export Results

In [ ]:
# Generate all plots
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

plots = generate_all_plots(scores_df, cases_df, FIGURES_DIR)
print(f"\nGenerated plots:")
for name, path in plots.items():
    print(f"  - {name}: {path}")

In [ ]:
# Save leaderboard
leaderboard.to_csv('../outputs/results/leaderboard.csv')
print("Saved leaderboard to outputs/results/leaderboard.csv")

# Save module breakdown
module_acc.to_csv('../outputs/results/module_accuracy.csv')
print("Saved module accuracy to outputs/results/module_accuracy.csv")

In [ ]:
# Generate failure report (if we have actual scores)
if scores:
    report = generate_failure_report(cases, scores)
    
    with open('../outputs/results/failure_report.md', 'w') as f:
        f.write(report)
    
    print("Saved failure report to outputs/results/failure_report.md")
    print("\n" + "="*50 + "\n")
    print(report[:2000] + "...")

## Summary

### Key Findings

1. **Overall Accuracy**: [Fill based on results]

2. **Module Performance**: 
   - Models tend to perform best on [module]
   - Weakest performance on [module]

3. **Difficulty Scaling**:
   - Easy cases: ~X% accuracy
   - Medium cases: ~Y% accuracy  
   - Hard cases: ~Z% accuracy

4. **Prompt Impact**:
   - Finance auditor prompt vs minimal prompt: [comparison]

5. **Common Failure Modes**:
   - [List key failure patterns]

### Recommendations

1. LLMs [can/cannot] reliably detect point-in-time validity issues
2. Strongest performance on [category]
3. Human oversight recommended for [category]
4. Prompt engineering [does/does not] significantly improve performance